In [1]:
# If ipywidgets is missing, run once:  !pip install ipywidgets
import sqlite3, math
from contextlib import closing
from datetime import date
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

DB_NAME = "snack_company.db"

In [2]:
def get_conn():
    conn = sqlite3.connect(DB_NAME)
    conn.execute("PRAGMA foreign_keys = ON")   # SQLite ignores FKs unless this is on
    return conn

def create_tables():
    with closing(get_conn()) as conn, conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS brands (
            brand_id         INTEGER PRIMARY KEY AUTOINCREMENT,
            brand_name       TEXT NOT NULL UNIQUE,
            supervisor_email TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS production_batches (
            batch_id         INTEGER PRIMARY KEY AUTOINCREMENT,
            brand_id         INTEGER NOT NULL,
            snack_name       TEXT NOT NULL,
            manufacture_date TEXT NOT NULL,
            FOREIGN KEY (brand_id) REFERENCES brands(brand_id) ON DELETE CASCADE
        );
        CREATE TABLE IF NOT EXISTS packing_items (
            item_id          INTEGER PRIMARY KEY AUTOINCREMENT,
            batch_id         INTEGER NOT NULL,
            item_description TEXT NOT NULL,
            cost_amount      REAL NOT NULL CHECK (cost_amount >= 0),
            FOREIGN KEY (batch_id) REFERENCES production_batches(batch_id) ON DELETE CASCADE
        );
        """)

SAMPLE = [
    ("Crunchy Munch", "crunchy.supervisor@snackco.com", [
        ("Masala Potato Chips", "2026-10-01", [("Foil Pouch", 4.50), ("Seasoning Mix", 12.75), ("Frying Oil", 30.00)]),
        ("Salted Potato Chips", "2026-10-05", [("Foil Pouch", 4.50), ("Frying Oil", 28.00)]),
    ]),
    ("Cheesy Bites", "cheesy.supervisor@snackco.com", [
        ("Cheese Nachos", "2026-10-03", [("Carton Box", 6.25), ("Cheese Powder", 15.00)]),
    ]),
]

def seed_if_empty():
    with closing(get_conn()) as conn, conn:
        if conn.execute("SELECT COUNT(*) FROM brands").fetchone()[0] > 0:
            return
        for brand, email, batches in SAMPLE:
            bid = conn.execute("INSERT INTO brands (brand_name, supervisor_email) VALUES (?,?)",
                               (brand, email)).lastrowid
            for snack, mdate, items in batches:
                batch_id = conn.execute(
                    "INSERT INTO production_batches (brand_id, snack_name, manufacture_date) VALUES (?,?,?)",
                    (bid, snack, mdate)).lastrowid
                conn.executemany(
                    "INSERT INTO packing_items (batch_id, item_description, cost_amount) VALUES (?,?,?)",
                    [(batch_id, d, c) for d, c in items])

create_tables()
seed_if_empty()

In [3]:
def query_rows(sql):
    with closing(get_conn()) as conn:
        return conn.execute(sql).fetchall()

def query_df(sql):
    with closing(get_conn()) as conn:
        return pd.read_sql_query(sql, conn)

def add_record(brand, snack, item, cost):
    with closing(get_conn()) as conn, conn:
        row = conn.execute("SELECT brand_id FROM brands WHERE brand_name = ? COLLATE NOCASE",
                           (brand,)).fetchone()
        if row:                                   # existing brand -> reuse ID
            brand_id = row[0]
        else:                                     # new brand -> create profile
            email = brand.lower().replace(" ", ".") + ".supervisor@snackco.com"
            brand_id = conn.execute("INSERT INTO brands (brand_name, supervisor_email) VALUES (?,?)",
                                    (brand, email)).lastrowid
        batch_id = conn.execute(
            "INSERT INTO production_batches (brand_id, snack_name, manufacture_date) VALUES (?,?,?)",
            (brand_id, snack, date.today().isoformat())).lastrowid
        conn.execute("INSERT INTO packing_items (batch_id, item_description, cost_amount) VALUES (?,?,?)",
                     (batch_id, item, cost))

DELETE_SQL = {
    "Brand":            "DELETE FROM brands WHERE brand_id = ?",
    "Production Batch": "DELETE FROM production_batches WHERE batch_id = ?",
    "Packing Item":     "DELETE FROM packing_items WHERE item_id = ?",
}

def delete_record(kind, record_id):
    with closing(get_conn()) as conn, conn:
        conn.execute(DELETE_SQL[kind], (record_id,))

def parse_cost(text):
    try:
        v = float(text.strip())
    except ValueError:
        return None
    return v if math.isfinite(v) and v >= 0 else None

JOIN_SQL = """
SELECT b.batch_id AS "Batch ID", br.brand_name AS "Brand Name", b.snack_name AS "Snack Name",
       b.manufacture_date AS "Manufacture Date", p.item_description AS "Packing Item",
       p.cost_amount AS "Cost Amount"
FROM production_batches b
JOIN brands br ON br.brand_id = b.brand_id
LEFT JOIN packing_items p ON p.batch_id = b.batch_id
ORDER BY b.batch_id
"""

In [4]:
# ---------- Insert panel ----------
brand_in = widgets.Text(description="Brand:",  placeholder="e.g. Crunchy Munch")
snack_in = widgets.Text(description="Snack:",  placeholder="e.g. Masala Potato Chips")
item_in  = widgets.Text(description="Item:",   placeholder="e.g. Foil Pouch")
cost_in  = widgets.Text(description="Cost:",   placeholder="e.g. 12.50")
add_btn  = widgets.Button(description="Add Record", button_style="success")

# ---------- Delete panel ----------
delete_type = widgets.Dropdown(options=["Brand", "Production Batch", "Packing Item"], description="Delete:")
target_dd   = widgets.Dropdown(description="Select:", layout=widgets.Layout(width="480px"))
del_btn     = widgets.Button(description="Delete Selected", button_style="danger")

status = widgets.HTML("")

# ---------- Tabs ----------
outs = [widgets.Output() for _ in range(4)]
tabs = widgets.Tab(children=outs)
for i, t in enumerate(["Combined View", "Brands", "Production Batches", "Packing Items"]):
    tabs.set_title(i, t)

def msg(text, ok=True):
    status.value = f"<b style='color:{'green' if ok else 'red'}'>{text}</b>"

def refresh_tables():
    queries = [JOIN_SQL, "SELECT * FROM brands", "SELECT * FROM production_batches",
               "SELECT * FROM packing_items"]
    for out, q in zip(outs, queries):
        df = query_df(q)
        with out:
            clear_output(wait=True)
            display(df)

def load_targets():
    kind = delete_type.value
    if kind == "Brand":
        rows = query_rows("SELECT brand_id, brand_name FROM brands ORDER BY brand_id")
        opts = [(f"{r[0]} - {r[1]}", r[0]) for r in rows]
    elif kind == "Production Batch":
        rows = query_rows("""SELECT b.batch_id, br.brand_name, b.snack_name FROM production_batches b
                             JOIN brands br ON br.brand_id = b.brand_id ORDER BY b.batch_id""")
        opts = [(f"Batch {r[0]} - {r[1]} / {r[2]}", r[0]) for r in rows]
    else:
        rows = query_rows("SELECT item_id, item_description, cost_amount, batch_id FROM packing_items ORDER BY item_id")
        opts = [(f"Item {r[0]} - {r[1]} ({r[2]}) [Batch {r[3]}]", r[0]) for r in rows]
    target_dd.options = opts
    target_dd.value = opts[0][1] if opts else None   # never points at a deleted ID

def refresh_all():
    refresh_tables()
    load_targets()

def on_add(_):
    brand, snack, item = brand_in.value.strip(), snack_in.value.strip(), item_in.value.strip()
    if not brand or not snack or not item or not cost_in.value.strip():
        return msg("All fields are required.", False)
    cost = parse_cost(cost_in.value)
    if cost is None:
        return msg("Cost must be a non-negative number.", False)
    add_record(brand, snack, item, cost)
    brand_in.value = snack_in.value = item_in.value = cost_in.value = ""
    refresh_all()
    msg("Record added successfully.")

def on_delete(_):
    if target_dd.value is None:
        return msg("Nothing to delete.", False)
    kind = delete_type.value
    delete_record(kind, target_dd.value)
    refresh_all()
    msg(f"{kind} deleted.")

add_btn.on_click(on_add)
del_btn.on_click(on_delete)
delete_type.observe(lambda change: load_targets(), names="value")

ui = widgets.VBox([
    widgets.HBox([
        widgets.VBox([widgets.HTML("<h3>Insert Panel</h3>"), brand_in, snack_in, item_in, cost_in, add_btn]),
        widgets.VBox([widgets.HTML("<h3>Delete Panel</h3>"), delete_type, target_dd, del_btn]),
    ]),
    status,
    tabs,
])

refresh_all()
display(ui)

In [5]:
with closing(get_conn()) as conn:
    for t in ["production_batches", "packing_items"]:
        print(t)
        for row in conn.execute(f"PRAGMA foreign_key_list({t})"):
            print("  references:", row[2], "| on_delete:", row[6])

production_batches
  references: brands | on_delete: CASCADE
packing_items
  references: production_batches | on_delete: CASCADE
